In [ ]:
import torch
from torch import nn, optim
from torchvision import datasets,transforms
from torch.utils.data import DataLoader
import torch.nn.functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

In [ ]:
transform_train = transforms.Compose([
    transforms.RandomResizedCrop(
        64,
        scale=(0.6, 1.0)
    ),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.RandomRotation(15),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.05
    ),

    transforms.ToTensor(),

    transforms.RandomErasing(
        p=0.25,
        scale=(0.02, 0.15)
    ),

    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])

transform_test = transforms.Compose([
    transforms.Resize((64,64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])

In [ ]:
train_data = datasets.ImageFolder(
    root="/kaggle/input/datasets/utkarshsaxenadn/fast-food-classification-dataset/Fast Food Classification V2/Train",
    transform=transform_train
)

test_data = datasets.ImageFolder(
    root="/kaggle/input/datasets/utkarshsaxenadn/fast-food-classification-dataset/Fast Food Classification V2/Valid",
    transform=transform_test
)

In [ ]:
train_dataload = DataLoader(
    train_data,batch_size=64,shuffle=True,num_workers=2, pin_memory=True
)
test_dataload = DataLoader(
    test_data,batch_size=64,shuffle=False,num_workers=2,pin_memory=True
)

In [ ]:
class SqueezeE(nn.Module):
    def __init__(self,in_c):
        super().__init__()

        r = 8
        reduced_dim = max(1,in_c // r)
        
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),

            nn.Conv2d(in_c,reduced_dim,1),
            nn.SiLU(),

            nn.Conv2d(reduced_dim,in_c,1),
            nn.Sigmoid()

        )
    def forward(self,x):
        return x * self.se(x)


        
class Block(nn.Module):
    def __init__(self,in_c,out_c,stride=1,expand_ratio=6):
        super().__init__()

        hidden_dim =in_c * expand_ratio
        
        self.se = SqueezeE(hidden_dim)
        self.conv_block = nn.Sequential(
            #Expandsion
            nn.Conv2d(in_c,hidden_dim,kernel_size=1,bias=False),
            nn.BatchNorm2d(hidden_dim),
            nn.SiLU(),

            #Depth wise
            nn.Conv2d(hidden_dim,hidden_dim,kernel_size=3,
                      stride=stride,
                      padding=1,
                      groups=hidden_dim,
                      bias=False),
            nn.BatchNorm2d(hidden_dim),
            nn.SiLU(),

           self.se,
            

            #Projection
            nn.Conv2d(hidden_dim,out_c,kernel_size=1,bias=False),
            nn.BatchNorm2d(out_c)
        )

        self.skip = nn.Identity()

        if stride != 1 or in_c != out_c:
            self.skip = nn.Sequential(
                nn.Conv2d(in_c, out_c, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_c)
            )
        self.act = nn.SiLU()

    def forward(self,x):
        return self.act(self.conv_block(x) + self.skip(x))
    
class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        

        self.convolutions = nn.Sequential(
            Block(3,32,stride=1),
            
            Block(32,64,stride=2),
            Block(64,64,stride=1),
            
            Block(64,128,stride=2),
            Block(128,128,stride=1),
            
            Block(128,256,stride=2),
            Block(256,256,stride=1),
            
            Block(256,512,stride=1)
        )

        self.final_layer = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.1),
            nn.Linear(512,10)
        )

    def forward(self,x):
        x = self.convolutions(x)
        x = self.final_layer(x)
        return x

In [ ]:
from torch.amp import autocast, GradScaler
max_epochs = 40
net = CNN().to(device) #or this 
loss_function = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(net.parameters(),lr = 5e-4,weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=max_epochs)

In [ ]:
import torch
import numpy as np

def mixup_data(x, y, alpha=0.4):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1

    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)

    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]

    return mixed_x, y_a, y_b, lam

In [ ]:
def mixup_loss(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
scaler = GradScaler()

best_acc = 0.0

train_losses = []
val_losses = []
val_accuracies = []

for epoch in range(max_epochs):

    net.train()

    print(f"Training Epoch {epoch}...")

    running_loss = 0.0

    for inputs, labels in train_dataload:

        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)
        # 🔥 APPLY MIXUP
        inputs, targets_a, targets_b, lam = mixup_data(inputs, labels, alpha=0.4)

        with autocast(device_type="cuda"):

            outputs = net(inputs)
            loss = lam * loss_function(outputs, targets_a) + \
               (1 - lam) * loss_function(outputs, targets_b)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_dataload)
    train_losses.append(epoch_loss)

    scheduler.step()

    # ---------------- VALIDATION ---------------- #

    net.eval()

    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for inputs, labels in test_dataload:

            inputs = inputs.to(device)
            labels = labels.to(device)

            with autocast(device_type="cuda"):

                outputs = net(inputs)
                loss = loss_function(outputs, labels)

            val_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    avg_val_loss = val_loss / len(test_dataload)
    val_acc = 100 * correct / total

    val_losses.append(avg_val_loss)
    val_accuracies.append(val_acc)


    print(f"Train Loss: {epoch_loss:.2f}")
    print(f"Val Loss: {avg_val_loss:.2f}")
    print(f"Val Accuracy: {val_acc:.2f}%")

In [ ]:
correct = 0
total = 0

net.eval()

with torch.no_grad():
    for data in test_dataload:
        images,labels = data
        images, labels = images.to(device), labels.to(device) 
        outputs = net(images)
        _,predicted = torch.max(outputs,1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Accuracy: {accuracy}%")

In [ ]:
model_data = {
    "model_state_dict" : net.state_dict(),
    "class_to_idx": train_data.class_to_idx
}
import torch

torch.save(model_data, "/kaggle/working/foodv2_model.pth")